In [1]:
#!/usr/bin/env python
# coding: utf-8

# # Composite DNA Decoder: Training & Evaluation - Eta-Based Variable Ratio
# ## Extended alphabet with variable mixture ratios (η=0.2, ℓ∈{-2,-1,0,1,2})

# =============================================================================
# CELL 1: DEVICE CONFIGURATION
# =============================================================================
import os
import torch

DEVICE_ID = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID


# =============================================================================
# CELL 2: IMPORTS
# =============================================================================
import random
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")


# =============================================================================
# CELL 3: CONFIGURATION & HYPERPARAMETERS
# =============================================================================

# ------------------- SELECT ERROR MODEL -------------------
ERROR_MODEL = "organick"

# ------------------- ETA-BASED ALPHABET PARAMETERS -------------------
ETA = 0.2
ELL_VALUES = [-2, -1, 0, 1, 2]

# Dataset parameters
NUM_SAMPLES = 100000
MAX_COVERAGE = 50

# Calculate vocab size
NUM_PURE_BASES = 4
NUM_TWO_MIX_PAIRS = 6
VOCAB_SIZE = NUM_PURE_BASES + NUM_TWO_MIX_PAIRS * len(ELL_VALUES)  # 34

# Error model specifications
ERROR_MODEL_SPECS = {
    "erlich": {"seq_length": 136, "name": "EZ17"},
    "grass": {"seq_length": 104, "name": "G15"},
    "organick": {"seq_length": 77, "name": "O17"}
}

# Build configuration
CONFIG = {
    # Error Model
    "error_model": ERROR_MODEL,
    "error_name": ERROR_MODEL_SPECS[ERROR_MODEL]["name"],
    
    # Eta Parameters
    "eta": ETA,
    "ell_values": ELL_VALUES,
    "alphabet_mode": f"eta{ETA}",
    
    # Data Paths
    "dataset_dir": "./dataset",
    "dataset_name": f"dna_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_eta{ETA}",
    
    # Results directory
    "results_dir": f"./results_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_eta{ETA}",
    
    # Vocabulary
    "vocab_size": VOCAB_SIZE,
    
    # Sequence Parameters
    "seq_length": ERROR_MODEL_SPECS[ERROR_MODEL]["seq_length"],
    
    # Experiment Parameters
    "coverage_levels": [1, 2, 3, 5, 8, 10, 15, 20, 25, 30, 40, 50],
    
    # Model Architecture
    "input_channels": 4,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": True,
    
    # Training Parameters
    "batch_size": 500,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 100,
    "patience": 10,
    "warmup_epochs": 10,
    "min_lr": 1e-6,
    
    # Reproducibility
    "seed": 42
}

# Complete dataset path
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"{CONFIG['dataset_name']}_"
                          f"{NUM_SAMPLES}_{MAX_COVERAGE}.pkl")

# Create results directory
os.makedirs(CONFIG['results_dir'], exist_ok=True)

print(f"{'='*60}")
print(f"📋 CONFIGURATION")
print(f"{'='*60}")
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_name']})")
print(f"   Eta: {CONFIG['eta']}, Ell Values: {CONFIG['ell_values']}")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Vocab Size: {CONFIG['vocab_size']} classes")
print(f"   Theoretical Capacity: {np.log2(CONFIG['vocab_size']):.4f} bits/position")
print(f"   Dataset Path: {CONFIG['dataset_path']}")
print(f"   Results Dir: {CONFIG['results_dir']}")
print(f"{'='*60}")


# =============================================================================
# CELL 4: SEED & REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    """Set seed for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")


# =============================================================================
# CELL 5: BUILD ETA-BASED ALPHABET MAPPINGS
# =============================================================================

def build_eta_based_symbol_to_idx(eta, ell_values):
    """Build symbol-to-index mapping for eta-based alphabet."""
    symbol_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    
    pair_names = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6']
    current_idx = 4
    
    for pair_name in pair_names:
        for ell in ell_values:
            if ell >= 0:
                symbol_name = f"{pair_name}_ell{ell}"
            else:
                symbol_name = f"{pair_name}_ell_neg{abs(ell)}"
            symbol_to_idx[symbol_name] = current_idx
            current_idx += 1
    
    return symbol_to_idx


def build_eta_based_ideal_vectors(eta, ell_values):
    """
    Build ideal frequency vectors for eta-based alphabet.
    
    Returns:
        torch.Tensor of shape (vocab_size, 4)
    """
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
    ]
    
    # Two-mix pairs: (vec_idx1, vec_idx2)
    pair_indices = [
        (0, 1),  # B1: A|C
        (0, 2),  # B2: A|G
        (0, 3),  # B3: A|T
        (1, 2),  # B4: C|G
        (1, 3),  # B5: C|T
        (2, 3),  # B6: G|T
    ]
    
    for idx1, idx2 in pair_indices:
        for ell in ell_values:
            prob1 = 0.5 + ell * eta
            prob2 = 0.5 - ell * eta
            
            vec = [0.0, 0.0, 0.0, 0.0]
            vec[idx1] = prob1
            vec[idx2] = prob2
            ideal_vectors.append(vec)
    
    return torch.tensor(ideal_vectors, dtype=torch.float32)


# Build mappings
SYMBOL_TO_IDX = build_eta_based_symbol_to_idx(CONFIG["eta"], CONFIG["ell_values"])
IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
IDEAL_VECTORS = build_eta_based_ideal_vectors(CONFIG["eta"], CONFIG["ell_values"]).to(device)

print(f"\n📊 Symbol Mappings (η={CONFIG['eta']}):")
print(f"   Total symbols: {len(SYMBOL_TO_IDX)}")
print(f"\n   Sample mappings (first 10):")
for i, (sym, idx) in enumerate(sorted(SYMBOL_TO_IDX.items(), key=lambda x: x[1])[:10]):
    vec = IDEAL_VECTORS[idx].cpu().numpy()
    print(f"   {sym:<16} {idx:<4} [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}, {vec[3]:.2f}]")
print(f"   ... ({len(SYMBOL_TO_IDX) - 10} more)")


# =============================================================================
# CELL 6: DATA PREPROCESSING (Same as previous)
# =============================================================================
# def preprocess_cluster_to_matrix(cluster_reads, target_length): ...
# Copy from previous code

def preprocess_cluster_to_matrix(cluster_reads, target_length):
    """
    Convert variable-length noisy reads into a (4, target_length) normalized frequency matrix.
    """
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    
    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
            
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            
            base = read[read_idx]
            if base in base_map:
                row_idx = base_map[base]
                profile_matrix[row_idx, t_idx] += 1.0
                
    if num_reads > 0:
        profile_matrix /= num_reads
        
    return profile_matrix


# =============================================================================
# CELL 7: PYTORCH DATASET CLASS (Same structure, updated for eta)
# =============================================================================

class CompositeDNADatasetEta(Dataset):
    """PyTorch Dataset for Eta-Based Composite DNA data."""
    
    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw_data = pickle.load(f)
        self.samples = raw_data['data']
        self.metadata = raw_data['metadata']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        
        if self.limit_coverage is not None:
            actual_limit = min(self.limit_coverage, len(cluster))
            cluster = cluster[:actual_limit]
            
        x_data = preprocess_cluster_to_matrix(cluster, self.seq_length)
        label_seq = item['label']
        y_data = np.array([self.symbol_to_idx[s] for s in label_seq], dtype=np.longlong)
        
        return torch.tensor(x_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)


# =============================================================================
# CELL 8: NEURAL NETWORK MODEL (Same as previous)
# =============================================================================
# class CompositeDecoderLSTM(nn.Module): ...
# Copy from previous code

class CompositeDecoderLSTM(nn.Module):
    """Bidirectional LSTM Decoder for Composite DNA."""
    
    def __init__(self, config):
        super(CompositeDecoderLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0
        )
        
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])
        
    def forward(self, x):
        # x: (Batch, 4, L) -> (Batch, L, 4)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        # Return: (Batch, vocab_size, L)
        return logits.permute(0, 2, 1)


# =============================================================================
# CELL 9: BASELINE DECODERS (Same logic, works with any ideal_vectors)
# =============================================================================
# def min_distance_decoder(obs, ideal_vectors): ...
# def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01): ...
# def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01): ...
# Copy from previous code

def min_distance_decoder(obs, ideal_vectors):
    """Minimum Euclidean Distance Decoder (L2 norm)."""
    dists = torch.sum((obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)


def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01):
    """KL Divergence Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmin(cross_entropy, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01):
    """Maximum Likelihood Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    
    log_likelihood = (obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmax(log_likelihood, dim=-1)


# =============================================================================
# CELL 10: EARLY STOPPING CLASS (Same as previous)
# =============================================================================
# class EarlyStopping: ...
# Copy from previous code

class EarlyStopping:
    """Early stopping with patience and best model saving."""
    
    def __init__(self, patience=5, path='checkpoint.pt', verbose=True):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.path = path
        self.verbose = verbose
        self.best_val_loss = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss
        
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score:
            self.counter += 1
            if self.verbose:
                print(f"      EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
            
    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f"      ✓ Val loss improved ({self.best_val_loss:.4f} → {val_loss:.4f}). Saving...")
        torch.save(model.state_dict(), self.path)
        self.best_val_loss = val_loss


# =============================================================================
# CELL 11: TRAINING FUNCTION (Same as previous)
# =============================================================================
# def train_model(model, train_loader, val_loader, config, weights_path, device): ...
# Copy from previous code

def train_model(model, train_loader, val_loader, config, weights_path, device):
    """Train the model with warmup + cosine annealing scheduler."""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=config['warmup_epochs'])
    cosine_scheduler = CosineAnnealingLR(
        optimizer, T_max=config['epochs'] - config['warmup_epochs'], eta_min=config['min_lr']
    )
    scheduler = SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[config['warmup_epochs']]
    )
    
    early_stopper = EarlyStopping(patience=config['patience'], path=weights_path, verbose=True)
    
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    
    print(f"\n   🏋️ Training Configuration:")
    print(f"      Epochs: {config['epochs']}, Patience: {config['patience']}")
    print(f"      Warmup: {config['warmup_epochs']} epochs")
    print(f"      LR: {config['learning_rate']} → {config['min_lr']}")
    
    for epoch in range(config['epochs']):
        start_time = time.time()
        
        # Training
        model.train()
        train_loss_accum = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss_accum += loss.item()
        avg_train_loss = train_loss_accum / len(train_loader)
        
        # Validation
        model.eval()
        val_loss_accum = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss_accum += criterion(outputs, labels).item()
        avg_val_loss = val_loss_accum / len(val_loader)
        
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['lr'].append(current_lr)
        
        elapsed = time.time() - start_time
        print(f"   Epoch {epoch+1:03d}/{config['epochs']} | "
              f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
              f"LR: {current_lr:.2e} | Time: {elapsed:.1f}s")
        
        scheduler.step()
        early_stopper(avg_val_loss, model)
        
        if early_stopper.early_stop:
            print(f"\n   🛑 Early stopping triggered at epoch {epoch+1}")
            break
    
    return history


# =============================================================================
# CELL 12: EVALUATION FUNCTION (Same as previous)
# =============================================================================
# def evaluate_all_decoders(model, loader, ideal_vectors, device): ...
# Copy from previous code

def evaluate_all_decoders(model, loader, ideal_vectors, device):
    """Evaluate all 4 decoders on the given data loader."""
    model.eval()
    
    correct = {'lstm': 0, 'mindist': 0, 'kl': 0, 'ml': 0}
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            obs = inputs.permute(0, 2, 1)
            
            outputs = model(inputs)
            pred_lstm = torch.argmax(outputs, dim=1)
            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            pred_kl = kl_divergence_decoder(obs, ideal_vectors)
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors)
            
            total += labels.numel()
            correct['lstm'] += (pred_lstm == labels).sum().item()
            correct['mindist'] += (pred_mindist == labels).sum().item()
            correct['kl'] += (pred_kl == labels).sum().item()
            correct['ml'] += (pred_ml == labels).sum().item()
    
    accuracies = {k: 100 * v / total for k, v in correct.items()}
    return accuracies


# =============================================================================
# CELL 13: FULL EXPERIMENT FOR SINGLE COVERAGE
# =============================================================================

def run_experiment_for_coverage(coverage_M, config, symbol_to_idx, ideal_vectors, device):
    """Run complete experiment for a single coverage level."""
    
    print(f"\n{'='*70}")
    print(f"🔬 EXPERIMENT FOR COVERAGE M = {coverage_M}")
    print(f"   Eta: {config['eta']}, Vocab Size: {config['vocab_size']}")
    print(f"{'='*70}")
    
    set_seed(config['seed'])
    
    full_ds = CompositeDNADatasetEta(
        config['dataset_path'], config['seq_length'], symbol_to_idx, limit_coverage=coverage_M
    )
    
    train_size = int(0.8 * len(full_ds))
    val_size = len(full_ds) - train_size
    train_ds, val_ds = random_split(full_ds, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=False, num_workers=0)
    
    print(f"   📊 Data: {train_size:,} train | {val_size:,} validation")
    
    model = CompositeDecoderLSTM(config).to(device)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🧠 Model: {config['vocab_size']} classes, {num_params:,} parameters")
    
    model_prefix = f"{config['error_name']}_eta{config['eta']}"
    best_weights_path = os.path.join(config['results_dir'], f"best_model_{model_prefix}_M{coverage_M}.pth")
    final_weights_path = os.path.join(config['results_dir'], f"final_model_{model_prefix}_M{coverage_M}.pth")
    history_path = os.path.join(config['results_dir'], f"training_history_{model_prefix}_M{coverage_M}.json")
    
    history = train_model(model, train_loader, val_loader, config, best_weights_path, device)
    
    torch.save(model.state_dict(), final_weights_path)
    print(f"   💾 Final model saved: {final_weights_path}")
    
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=4)
    
    print(f"\n   📈 Evaluating all decoders...")
    model.load_state_dict(torch.load(best_weights_path, map_location=device))
    
    accuracies = evaluate_all_decoders(model, val_loader, ideal_vectors, device)
    
    print(f"\n   ✅ RESULTS M={coverage_M} (η={config['eta']}):")
    print(f"      Bi-LSTM:         {accuracies['lstm']:.2f}%")
    print(f"      Min. Distance:   {accuracies['mindist']:.2f}%")
    print(f"      KL Divergence:   {accuracies['kl']:.2f}%")
    print(f"      Max. Likelihood: {accuracies['ml']:.2f}%")
    
    return accuracies, history


# =============================================================================
# CELL 14: PLOTTING FUNCTIONS (Same as previous)
# =============================================================================
# def plot_training_history(history, coverage_M, save_path, config): ...
# def plot_comparison_results(results, save_path, config): ...
# Copy from previous code

def plot_training_history(history, coverage_M, save_path, config):
    """Plot training and validation loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val Loss')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'Training & Validation Loss (M={coverage_M}, η={config["eta"]})', fontsize=14)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, history['lr'], 'g-', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Learning Rate', fontsize=12)
    ax2.set_title(f'Learning Rate Schedule (M={coverage_M})', fontsize=14)
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   📈 Training plot saved: {save_path}")


def plot_comparison_results(results, save_path, config):
    """Plot comparison of all decoders across coverage levels."""
    plt.figure(figsize=(12, 7))
    
    plt.plot(results['coverage'], results['lstm'], 
             'o-', lw=2.5, ms=8, c='#2ecc71', label='Bi-LSTM (Ours)')
    plt.plot(results['coverage'], results['mindist'], 
             's--', lw=2.5, ms=8, c='#e74c3c', label='Min. Distance')
    plt.plot(results['coverage'], results['kl'], 
             '^-.', lw=2.5, ms=8, c='#3498db', label='KL Divergence')
    plt.plot(results['coverage'], results['ml'], 
             'd:', lw=2.5, ms=8, c='#9b59b6', label='Max. Likelihood')
    
    title = (f"Composite DNA Decoding: η={config['eta']} ({config['vocab_size']} classes)\n"
             f"Error Model: {config['error_name']}, Seq Length: {config['seq_length']}")
    
    plt.xlabel("Coverage Depth (M)", fontsize=12)
    plt.ylabel("Symbol Accuracy (%)", fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(fontsize=11, loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 105)
    plt.xticks(results['coverage'])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Comparison plot saved: {save_path}")


# =============================================================================
# CELL 15: VERIFY DATASET EXISTS
# =============================================================================

print("\n" + "="*70)
print("📦 LOADING DATASET")
print("="*70)

if not os.path.exists(CONFIG['dataset_path']):
    raise FileNotFoundError(
        f"\n❌ Dataset not found: {CONFIG['dataset_path']}\n"
        f"   Please run dataset generator first with:\n"
        f"   ETA = {CONFIG['eta']}, ELL_VALUES = {CONFIG['ell_values']}"
    )

with open(CONFIG['dataset_path'], 'rb') as f:
    data = pickle.load(f)

print(f"✅ Dataset loaded: {CONFIG['dataset_path']}")
print(f"   Samples: {len(data['data']):,}")
print(f"   Vocab Size: {data['metadata']['vocab_size']}")
print(f"   Eta: {data['metadata']['eta']}")


# =============================================================================
# CELL 16: MAIN EXECUTION - RUN ALL EXPERIMENTS
# =============================================================================

print("\n" + "="*70)
print("🚀 RUNNING EXPERIMENTS FOR ALL COVERAGE LEVELS")
print("="*70)
print(f"   Eta: {CONFIG['eta']}, Vocab Size: {CONFIG['vocab_size']}")
print(f"   Coverage Levels: {CONFIG['coverage_levels']}")

results = {
    'coverage': CONFIG['coverage_levels'],
    'lstm': [],
    'mindist': [],
    'kl': [],
    'ml': [],
    'config': {
        'error_model': CONFIG['error_model'],
        'error_name': CONFIG['error_name'],
        'seq_length': CONFIG['seq_length'],
        'eta': CONFIG['eta'],
        'ell_values': CONFIG['ell_values'],
        'vocab_size': CONFIG['vocab_size'],
        'hidden_dim': CONFIG['hidden_dim'],
        'num_layers': CONFIG['num_layers'],
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
}

all_histories = {}

for M in CONFIG['coverage_levels']:
    accuracies, history = run_experiment_for_coverage(
        M, CONFIG, SYMBOL_TO_IDX, IDEAL_VECTORS, device
    )
    
    results['lstm'].append(accuracies['lstm'])
    results['mindist'].append(accuracies['mindist'])
    results['kl'].append(accuracies['kl'])
    results['ml'].append(accuracies['ml'])
    all_histories[M] = history
    
    plot_prefix = f"{CONFIG['error_name']}_eta{CONFIG['eta']}"
    plot_path = os.path.join(CONFIG['results_dir'], f"training_plot_{plot_prefix}_M{M}.png")
    plot_training_history(history, M, plot_path, CONFIG)


# =============================================================================
# CELL 17: SAVE FINAL RESULTS & PLOT
# =============================================================================

print("\n" + "="*70)
print("📊 FINAL RESULTS SUMMARY")
print("="*70)

results_json_path = os.path.join(CONFIG['results_dir'], "experiment_results.json")
with open(results_json_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f"💾 Results saved: {results_json_path}")

print(f"\n   Eta: {CONFIG['eta']}, Vocab Size: {CONFIG['vocab_size']}")
print(f"   {'M':<8} {'Bi-LSTM':<12} {'Min.Dist':<12} {'KL Div':<12} {'Max.Like':<12}")
print(f"   {'-'*56}")
for i, M in enumerate(results['coverage']):
    print(f"   {M:<8} {results['lstm'][i]:<12.2f} {results['mindist'][i]:<12.2f} "
          f"{results['kl'][i]:<12.2f} {results['ml'][i]:<12.2f}")
print(f"   {'='*56}")

plot_path = os.path.join(CONFIG['results_dir'], "final_comparison_plot.png")
plot_comparison_results(results, plot_path, CONFIG)

print(f"\n✅ All experiments completed!")
print(f"   Eta: {CONFIG['eta']}, Classes: {CONFIG['vocab_size']}")
print(f"   Results directory: {CONFIG['results_dir']}")

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080
📋 CONFIGURATION
   Error Model: organick (O17)
   Eta: 0.2, Ell Values: [-2, -1, 0, 1, 2]
   Sequence Length: 77
   Vocab Size: 34 classes
   Theoretical Capacity: 5.0875 bits/position
   Dataset Path: ./dataset/dna_O17_eta0.2_100000_50.pkl
   Results Dir: ./results_O17_eta0.2
🎲 Random seed set to: 42

📊 Symbol Mappings (η=0.2):
   Total symbols: 34

   Sample mappings (first 10):
   A                0    [1.00, 0.00, 0.00, 0.00]
   C                1    [0.00, 1.00, 0.00, 0.00]
   G                2    [0.00, 0.00, 1.00, 0.00]
   T                3    [0.00, 0.00, 0.00, 1.00]
   B1_ell_neg2      4    [0.10, 0.90, 0.00, 0.00]
   B1_ell_neg1      5    [0.30, 0.70, 0.00, 0.00]
   B1_ell0          6    [0.50, 0.50, 0.00, 0.00]
   B1_ell1          7    [0.70, 0.30, 0.00, 0.00]
   B1_ell2          8    [0.90, 0.10, 0.00, 0.00]
   B2_ell_neg2      9    [0.10, 0.00, 0.90, 0.00]
   ... (24 more)

📦 LOADING DATASET
✅ Dataset loaded: ./dataset

/homes/shubham/anaconda3/envs/pytorchenv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:149: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


   Epoch 011/100 | Train: 2.6697 | Val: 2.6663 | LR: 1.00e-03 | Time: 26.1s
      ✓ Val loss improved (2.6664 → 2.6663). Saving...
   Epoch 012/100 | Train: 2.6688 | Val: 2.6653 | LR: 1.00e-03 | Time: 25.0s
      ✓ Val loss improved (2.6663 → 2.6653). Saving...
   Epoch 013/100 | Train: 2.6684 | Val: 2.6656 | LR: 9.99e-04 | Time: 24.8s
      EarlyStopping counter: 1/10
   Epoch 014/100 | Train: 2.6679 | Val: 2.6657 | LR: 9.97e-04 | Time: 25.5s
      EarlyStopping counter: 2/10
   Epoch 015/100 | Train: 2.6677 | Val: 2.6646 | LR: 9.95e-04 | Time: 28.6s
      ✓ Val loss improved (2.6653 → 2.6646). Saving...
   Epoch 016/100 | Train: 2.6672 | Val: 2.6643 | LR: 9.92e-04 | Time: 26.6s
      ✓ Val loss improved (2.6646 → 2.6643). Saving...
   Epoch 017/100 | Train: 2.6670 | Val: 2.6644 | LR: 9.89e-04 | Time: 24.4s
      EarlyStopping counter: 1/10
   Epoch 018/100 | Train: 2.6669 | Val: 2.6641 | LR: 9.85e-04 | Time: 24.3s
      ✓ Val loss improved (2.6643 → 2.6641). Saving...
   Epoch 019/10

   Epoch 025/100 | Train: 2.1693 | Val: 2.1673 | LR: 9.42e-04 | Time: 47.6s
      ✓ Val loss improved (2.1679 → 2.1673). Saving...
   Epoch 026/100 | Train: 2.1687 | Val: 2.1675 | LR: 9.33e-04 | Time: 48.1s
      EarlyStopping counter: 1/10
   Epoch 027/100 | Train: 2.1684 | Val: 2.1676 | LR: 9.24e-04 | Time: 47.5s
      EarlyStopping counter: 2/10
   Epoch 028/100 | Train: 2.1681 | Val: 2.1669 | LR: 9.15e-04 | Time: 40.4s
      ✓ Val loss improved (2.1673 → 2.1669). Saving...
   Epoch 029/100 | Train: 2.1679 | Val: 2.1671 | LR: 9.05e-04 | Time: 40.3s
      EarlyStopping counter: 1/10
   Epoch 030/100 | Train: 2.1676 | Val: 2.1666 | LR: 8.94e-04 | Time: 40.4s
      ✓ Val loss improved (2.1669 → 2.1666). Saving...
   Epoch 031/100 | Train: 2.1672 | Val: 2.1666 | LR: 8.83e-04 | Time: 41.0s
      ✓ Val loss improved (2.1666 → 2.1666). Saving...
   Epoch 032/100 | Train: 2.1669 | Val: 2.1662 | LR: 8.72e-04 | Time: 40.4s
      ✓ Val loss improved (2.1666 → 2.1662). Saving...
   Epoch 033/10

   Epoch 030/100 | Train: 1.8823 | Val: 1.8814 | LR: 8.94e-04 | Time: 59.5s
      ✓ Val loss improved (1.8820 → 1.8814). Saving...
   Epoch 031/100 | Train: 1.8819 | Val: 1.8815 | LR: 8.83e-04 | Time: 60.7s
      EarlyStopping counter: 1/10
   Epoch 032/100 | Train: 1.8814 | Val: 1.8808 | LR: 8.72e-04 | Time: 57.1s
      ✓ Val loss improved (1.8814 → 1.8808). Saving...
   Epoch 033/100 | Train: 1.8810 | Val: 1.8813 | LR: 8.60e-04 | Time: 57.7s
      EarlyStopping counter: 1/10
   Epoch 034/100 | Train: 1.8809 | Val: 1.8804 | LR: 8.47e-04 | Time: 57.3s
      ✓ Val loss improved (1.8808 → 1.8804). Saving...
   Epoch 035/100 | Train: 1.8803 | Val: 1.8802 | LR: 8.35e-04 | Time: 57.2s
      ✓ Val loss improved (1.8804 → 1.8802). Saving...
   Epoch 036/100 | Train: 1.8798 | Val: 1.8805 | LR: 8.22e-04 | Time: 57.1s
      EarlyStopping counter: 1/10
   Epoch 037/100 | Train: 1.8796 | Val: 1.8800 | LR: 8.08e-04 | Time: 58.1s
      ✓ Val loss improved (1.8802 → 1.8800). Saving...
   Epoch 038/10

...

   Epoch 028/100 | Train: 0.4905 | Val: 0.4823 | LR: 9.15e-04 | Time: 492.9s
      ✓ Val loss improved (0.4828 → 0.4823). Saving...
   Epoch 029/100 | Train: 0.4894 | Val: 0.4818 | LR: 9.05e-04 | Time: 493.2s
      ✓ Val loss improved (0.4823 → 0.4818). Saving...
   Epoch 030/100 | Train: 0.4884 | Val: 0.4806 | LR: 8.94e-04 | Time: 493.5s
      ✓ Val loss improved (0.4818 → 0.4806). Saving...
   Epoch 031/100 | Train: 0.4874 | Val: 0.4803 | LR: 8.83e-04 | Time: 493.4s
      ✓ Val loss improved (0.4806 → 0.4803). Saving...
   Epoch 032/100 | Train: 0.4867 | Val: 0.4799 | LR: 8.72e-04 | Time: 493.5s
      ✓ Val loss improved (0.4803 → 0.4799). Saving...
   Epoch 033/100 | Train: 0.4860 | Val: 0.4787 | LR: 8.60e-04 | Time: 492.4s
      ✓ Val loss improved (0.4799 → 0.4787). Saving...
   Epoch 034/100 | Train: 0.4852 | Val: 0.4787 | LR: 8.47e-04 | Time: 492.5s
      EarlyStopping counter: 1/10
   Epoch 035/100 | Train: 0.4846 | Val: 0.4790 | LR: 8.35e-04 | Time: 493.0s
      EarlyStopping 

   Epoch 095/100 | Train: 0.4691 | Val: 0.4698 | LR: 1.19e-05 | Time: 503.9s
      ✓ Val loss improved (0.4698 → 0.4698). Saving...
   Epoch 096/100 | Train: 0.4691 | Val: 0.4698 | LR: 8.59e-06 | Time: 496.9s
      ✓ Val loss improved (0.4698 → 0.4698). Saving...
   Epoch 097/100 | Train: 0.4691 | Val: 0.4697 | LR: 5.86e-06 | Time: 497.7s
      ✓ Val loss improved (0.4698 → 0.4697). Saving...
   Epoch 098/100 | Train: 0.4691 | Val: 0.4698 | LR: 3.74e-06 | Time: 500.5s
      EarlyStopping counter: 1/10
   Epoch 099/100 | Train: 0.4690 | Val: 0.4697 | LR: 2.22e-06 | Time: 496.3s
      EarlyStopping counter: 2/10
   Epoch 100/100 | Train: 0.4690 | Val: 0.4697 | LR: 1.30e-06 | Time: 496.8s
      EarlyStopping counter: 3/10
   💾 Final model saved: ./results_O17_eta0.2/final_model_O17_eta0.2_M30.pth

   📈 Evaluating all decoders...

   ✅ RESULTS M=30 (η=0.2):
      Bi-LSTM:         81.57%
      Min. Distance:   76.87%
      KL Divergence:   77.53%
      Max. Likelihood: 77.53%
   📈 Training 

   Epoch 054/100 | Train: 0.3403 | Val: 0.3373 | LR: 5.35e-04 | Time: 656.0s
      EarlyStopping counter: 5/10
   Epoch 055/100 | Train: 0.3397 | Val: 0.3370 | LR: 5.18e-04 | Time: 656.0s
      ✓ Val loss improved (0.3373 → 0.3370). Saving...
   Epoch 056/100 | Train: 0.3393 | Val: 0.3360 | LR: 5.00e-04 | Time: 655.3s
      ✓ Val loss improved (0.3370 → 0.3360). Saving...
   Epoch 057/100 | Train: 0.3390 | Val: 0.3361 | LR: 4.83e-04 | Time: 657.4s
      EarlyStopping counter: 1/10
   Epoch 058/100 | Train: 0.3389 | Val: 0.3363 | LR: 4.66e-04 | Time: 656.9s
      EarlyStopping counter: 2/10
   Epoch 059/100 | Train: 0.3385 | Val: 0.3360 | LR: 4.48e-04 | Time: 658.5s
      EarlyStopping counter: 3/10
   Epoch 060/100 | Train: 0.3384 | Val: 0.3360 | LR: 4.31e-04 | Time: 658.4s
      EarlyStopping counter: 4/10
   Epoch 061/100 | Train: 0.3381 | Val: 0.3358 | LR: 4.14e-04 | Time: 658.6s
      ✓ Val loss improved (0.3360 → 0.3358). Saving...
   Epoch 062/100 | Train: 0.3378 | Val: 0.3354 | 

   Epoch 015/100 | Train: 0.2855 | Val: 0.2640 | LR: 9.95e-04 | Time: 891.2s
      ✓ Val loss improved (0.2668 → 0.2640). Saving...
   Epoch 016/100 | Train: 0.2814 | Val: 0.2601 | LR: 9.92e-04 | Time: 821.6s
      ✓ Val loss improved (0.2640 → 0.2601). Saving...
   Epoch 017/100 | Train: 0.2775 | Val: 0.2588 | LR: 9.89e-04 | Time: 816.4s
      ✓ Val loss improved (0.2601 → 0.2588). Saving...
   Epoch 018/100 | Train: 0.2749 | Val: 0.2591 | LR: 9.85e-04 | Time: 817.3s
      EarlyStopping counter: 1/10
   Epoch 019/100 | Train: 0.2731 | Val: 0.2565 | LR: 9.81e-04 | Time: 818.3s
      ✓ Val loss improved (0.2588 → 0.2565). Saving...
   Epoch 020/100 | Train: 0.2710 | Val: 0.2572 | LR: 9.76e-04 | Time: 817.1s
      EarlyStopping counter: 1/10
   Epoch 021/100 | Train: 0.2696 | Val: 0.2571 | LR: 9.70e-04 | Time: 817.3s
      EarlyStopping counter: 2/10
   Epoch 022/100 | Train: 0.2679 | Val: 0.2560 | LR: 9.64e-04 | Time: 819.3s
      ✓ Val loss improved (0.2565 → 0.2560). Saving...
   Epoc

   Epoch 082/100 | Train: 0.2432 | Val: 0.2420 | LR: 1.07e-04 | Time: 818.7s
      EarlyStopping counter: 1/10
   Epoch 083/100 | Train: 0.2431 | Val: 0.2420 | LR: 9.64e-05 | Time: 819.9s
      EarlyStopping counter: 2/10
   Epoch 084/100 | Train: 0.2431 | Val: 0.2418 | LR: 8.64e-05 | Time: 824.6s
      ✓ Val loss improved (0.2419 → 0.2418). Saving...
   Epoch 085/100 | Train: 0.2429 | Val: 0.2418 | LR: 7.69e-05 | Time: 818.2s
      EarlyStopping counter: 1/10
   Epoch 086/100 | Train: 0.2428 | Val: 0.2419 | LR: 6.79e-05 | Time: 820.0s
      EarlyStopping counter: 2/10
   Epoch 087/100 | Train: 0.2428 | Val: 0.2418 | LR: 5.95e-05 | Time: 819.7s
      EarlyStopping counter: 3/10
   Epoch 088/100 | Train: 0.2427 | Val: 0.2417 | LR: 5.16e-05 | Time: 816.8s
      ✓ Val loss improved (0.2418 → 0.2417). Saving...
   Epoch 089/100 | Train: 0.2426 | Val: 0.2417 | LR: 4.42e-05 | Time: 821.1s
      ✓ Val loss improved (0.2417 → 0.2417). Saving...
   Epoch 090/100 | Train: 0.2426 | Val: 0.2416 | 